# Three Transformer Architectures, in Three Small Demos

There are three basic "shapes" of transformer model, and this notebook lets you
run one tiny example of each:

| Part | Architecture | Task | Model |
|---|---|---|---|
| A | Encoder-Decoder | Machine translation | `facebook/m2m100_418M` |
| B | Decoder-Only | Next-token text generation | `gpt2` |
| C | Encoder-Only | Sentiment classification | `distilbert-base-uncased-finetuned-sst-2-english` |

**How to use this notebook:** run the cells from top to bottom, in order (click
each cell, press `Shift+Enter`). The first cell installs everything you need.
Full explanations of *why* each part works this way are in the separate
explainer document, not here — this notebook's comments are deliberately kept
short so the code stays readable.

In [ ]:
# Install the libraries this notebook needs.
# - transformers: gives us pretrained models + a simple pipeline() API
# - sentencepiece: the tokenizer library the translation model needs
# - accelerate: lets transformers place the model on GPU automatically
!pip install -q "transformers[torch]" sentencepiece accelerate

---
## Part A — Encoder-Decoder: Machine Translation

An encoder-decoder model has two halves: an **encoder** that reads and
understands the whole input sentence, and a **decoder** that writes the
output sentence one word at a time using what the encoder understood.
That's exactly what translation needs: read a full sentence in one
language, then produce a full sentence in another.

We use `facebook/m2m100_418M`, a model trained to translate directly
between 100 languages (no need for a separate model per language pair).

In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import torch

# Use the GPU if Colab gave us one (Runtime > Change runtime type > T4 GPU),
# otherwise fall back to CPU. Translation works either way, just slower on CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "facebook/m2m100_418M"

# The tokenizer turns text into the numeric IDs the model understands.
tokenizer = M2M100Tokenizer.from_pretrained(model_name)

# The model itself: this downloads ~1.9 GB of pretrained weights the
# first time you run this cell, then reuses them for the rest of the session.
model = M2M100ForConditionalGeneration.from_pretrained(model_name).to(device)

In [ ]:
def translate(text, src_lang, tgt_lang):
    """Translate `text` from `src_lang` to `tgt_lang` using M2M100.

    src_lang / tgt_lang are short language codes, e.g. "en", "fr", "hi".
    See the code table in the next cell for the ones this notebook uses.
    """
    # Tell the tokenizer what language the INPUT text is in.
    tokenizer.src_lang = src_lang

    # Turn the text into tensors (numeric arrays) the model can read.
    encoded = tokenizer(text, return_tensors="pt").to(device)

    # Ask the model to generate a translation.
    # forced_bos_token_id tells the decoder "start the OUTPUT in this language".
    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        max_new_tokens=128,   # never generate more than this many output tokens
        num_beams=5,          # beam search: explore 5 candidate translations, keep the best
    )

    # Turn the model's output tokens back into readable text.
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

### Common language codes

| Language | Code | | Language | Code |
|---|---|---|---|---|
| English | `en` | | Hindi | `hi` |
| French | `fr` | | Chinese | `zh` |
| Spanish | `es` | | Japanese | `ja` |
| German | `de` | | Arabic | `ar` |
| Kannada | `kn` | | Russian | `ru` |

Full list: [M2M100 supported languages](https://huggingface.co/facebook/m2m100_418M) (see model card).

In [ ]:
# A few fixed examples so you can see it working immediately.
print("EN -> FR:", translate("Hello, how are you today?", "en", "fr"))
print("EN -> HI:", translate("The weather is nice today.", "en", "hi"))
print("EN -> ES:", translate("I love programming.", "en", "es"))
print("FR -> EN:", translate("Bonjour, comment allez-vous ?", "fr", "en"))

In [ ]:
# Try it yourself: type any sentence and any target language code.
my_text = input("Enter a sentence in English: ")
my_target_lang = input("Enter the target language code (e.g. fr, hi, es, de): ")

print("\nTranslation:", translate(my_text, "en", my_target_lang))

---
## Part B — Decoder-Only: Next-Token Text Generation

A decoder-only model does one thing, over and over: given the text so far,
predict the single most likely **next token**. A whole paragraph is just that
one step repeated — predict a token, add it to the text, predict the next
one, and so on. There's no separate "encoder" reading a source sentence;
there's just a running context that keeps growing. GPT-2 is a small, classic
decoder-only model, good for seeing this directly.

In [ ]:
from transformers import pipeline

# Load the text-generation pipeline with a decoder-only GPT-2 model.
generator = pipeline(task="text-generation", model="gpt2", device=0 if device == "cuda" else -1)

# GPT-2 has no dedicated padding token by default; reuse its end-of-text
# token so the pipeline doesn't warn about it.
generator.tokenizer.pad_token_id = generator.tokenizer.eos_token_id

In [ ]:
prompt = "Artificial intelligence is going to"

output = generator(
    prompt,
    max_new_tokens=40,       # generate up to 40 NEW tokens after the prompt
    num_return_sequences=1,  # how many different continuations to generate
)

# `output` is a list of dicts; each dict has a "generated_text" key.
print(output[0]["generated_text"])

In [ ]:
# Try it yourself: give GPT-2 your own starting sentence and watch it continue.
my_prompt = input("Enter the start of a sentence: ")

my_output = generator(my_prompt, max_new_tokens=40, num_return_sequences=1)
print("\nGenerated:", my_output[0]["generated_text"])

---
## Part C — Encoder-Only: Sentiment Analysis

An encoder-only model reads the whole input at once and produces a single
understanding of it — there's no decoder generating new text. That's exactly
what classification needs: read a sentence, output one label (here:
POSITIVE or NEGATIVE) plus a confidence score. We use DistilBERT, a smaller
and faster version of BERT, already fine-tuned for sentiment.

In [ ]:
# Load the sentiment-analysis pipeline with an encoder-only DistilBERT model,
# already fine-tuned on movie review sentiment (SST-2).
classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if device == "cuda" else -1,
)

In [ ]:
# A few fixed examples: one clearly positive, one clearly negative,
# and a batch (list) of two more, to show it works on multiple sentences at once.
print(classifier("I absolutely loved this movie, it was fantastic!"))
print(classifier("This was a terrible waste of time."))
print(classifier(["Great product, highly recommend.", "I am so disappointed with this."]))

In [ ]:
# Try it yourself: type any sentence and see whether it's classified
# as POSITIVE or NEGATIVE, and how confident the model is (score 0-1).
my_sentence = input("Enter a sentence: ")
print(classifier(my_sentence))

---
## Manual sanity checks

Run this cell last. It re-runs one example from each part and checks the
output looks right. If all three lines print `PASSED`, the whole notebook
is working end to end.

In [ ]:
# --- Check A: translation produces non-empty, different-looking text ---
fr = translate("Good morning.", "en", "fr")
assert isinstance(fr, str) and len(fr) > 0 and fr.lower() != "good morning."
print("Part A (translation):", "PASSED", "->", fr)

# --- Check B: generation returns more text than just the prompt ---
gen = generator("The weather today is", max_new_tokens=15, num_return_sequences=1)[0]["generated_text"]
assert isinstance(gen, str) and len(gen) > len("The weather today is")
print("Part B (generation):", "PASSED", "->", gen)

# --- Check C: sentiment picks the right label on obvious examples ---
pos = classifier("I love this, it's wonderful!")[0]
neg = classifier("I hate this, it's awful.")[0]
assert pos["label"] == "POSITIVE" and neg["label"] == "NEGATIVE"
print("Part C (sentiment):", "PASSED", "->", pos, neg)

print("\nAll checks passed. The notebook is working correctly.")